# N01B - Audit the supplied extract `BDHS_Stunting_XAI_16Features_Meta.dta` (plan stage 1, extract mode)
**Use this notebook when you only have the extract. Use N01 only if you also have the original `BDKR81FL.DTA`.** Both write the same files, so N02-N04 work with either.

What is different from N01 (observed in the extract, not assumed)
- The extract has **no `hw70`, `b5`, `v135`, `v501` or `caseid`**. It has a ready-made binary column `stunting`.
- So this notebook **cannot** re-derive stunting from `hw70 < -200`, cannot apply the DHS special-code / validity rules, cannot rebuild the sample flow (U1) and cannot cross-tabulate code 97 with `v135` or `v701/v705` missingness with `v501`.
- It audits everything that *can* be checked from the file, and writes the cohort file the next notebooks need. Sign-off here means "I accept the extract as the **provisional** working cohort"; D1 and U1 stay open and are recorded as open in the sign-off file.

In [ ]:
# --- setup (same in every notebook) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab - using local folder.")
import os, sys, time
import numpy as np
import pandas as pd
BASE_DIR = os.environ.get("THESIS_BASE_DIR", "/content/drive/MyDrive/THESIS_V2")
sys.path.append(BASE_DIR + "/code")
import bdhs_helpers as h
h.make_all_folders()
h.set_all_seeds(h.SEED_SPLIT)
h.print_environment("N01B")

In [ ]:
EXTRACT_FILE = h.DATA_DIR + "/BDHS_Stunting_XAI_16Features_Meta.dta"
print("extract:", EXTRACT_FILE, "| sha256:", h.file_sha256(EXTRACT_FILE)[:16])
raw = pd.read_stata(EXTRACT_FILE, convert_categoricals=False)
print("shape (rows, columns):", raw.shape)
print("columns:", list(raw.columns))

REQUIRED = h.PREDICTORS_DEFAULT + ["v001", "v021", "v022", "v023", "v005", "stunting"]
missing_required = []
for col in REQUIRED:
    if col not in raw.columns:
        missing_required.append(col)
if len(missing_required) > 0:
    raise ValueError("Required columns missing from the extract: " + str(missing_required))

print()
print("Columns needed to rebuild the cohort / audit codes that are NOT in this file:")
for col in ["hw70", "b5", "v135", "v501", "caseid"]:
    print("  ", col, "->", "present" if col in raw.columns else "ABSENT")

cohort = raw.copy()
cohort["row_id"] = np.arange(len(cohort))     # stable key = row position in the extract

### Outcome as supplied
`stunting` was derived outside this notebook. Its definition (hw70 < -200 among valid HAZ) **cannot be verified from this file**.

In [ ]:
values = sorted(list(cohort["stunting"].unique()))
print("distinct values of stunting:", values)
if not (values == [0, 1]):
    raise ValueError("stunting must contain only 0 and 1")
cohort["stunted"] = cohort["stunting"].astype(int)
print("stunted:", int(cohort["stunted"].sum()), "of", len(cohort), "| prevalence (unweighted):", round(cohort["stunted"].mean(), 4))

flow = []
flow.append(["0 rows in supplied extract", "none", 0, len(cohort)])
flow.append(["1-6 age / eligibility / HAZ validity rules", "NOT REBUILT: hw70 and eligibility fields are not in the extract (U1, D1 open)", 0, len(cohort)])
flow_df = pd.DataFrame(flow, columns=["step", "rule", "n_removed", "n_remaining"])
print(flow_df.to_string(index=False))
print()
print("Plan v2.2 reference figures: 4,420 eligible for anthropometry; 4,199 mother interviewed; extract 4,105 (provisional).")
print("Child age range in extract (b19):", cohort["b19"].min(), "to", cohort["b19"].max(), "(expected 0-59)")

### Survey design variables

In [ ]:
print("v001 equals v021 in all rows:", bool((cohort["v001"] == cohort["v021"]).all()))
print("v022 equals v023 in all rows:", bool((cohort["v022"] == cohort["v023"]).all()))
print("Plan rule: PSU = v021, stratum = v022.")
print("PSUs:", cohort["v021"].nunique(), "| strata:", cohort["v022"].nunique())
rows_per_psu = cohort.groupby("v021").size()
print("children per PSU: min", rows_per_psu.min(), "| median", rows_per_psu.median(), "| max", rows_per_psu.max())
psu_per_stratum = cohort.groupby("v022")["v021"].nunique()
print("fewest PSUs in any stratum:", psu_per_stratum.min(), "| strata with fewer than 2 PSUs:", int((psu_per_stratum < 2).sum()))
if "v002" in cohort.columns:
    dup = int(cohort.duplicated(["v001", "v002"]).sum())
    print("children sharing a household with another child in the extract (v001+v002 repeats):", dup, "(PSU grouping already keeps households together)")

### U2 checks that the extract allows
The extract stores **no text value labels** (labels equal codes), so the meaning of codes such as 97, 8, 98 must come from the DHS recode manual.

In [ ]:
key_counts = []

b11_missing = cohort["b11"].isna()
first_birth = cohort["bord"] == 1
n_first = int((b11_missing & first_birth).sum())
n_later = int((b11_missing & ~first_birth).sum())
n_first_valid = int((~b11_missing & first_birth).sum())
print("b11: non-missing range", cohort["b11"].min(), "to", cohort["b11"].max(), "| distinct values:", cohort["b11"].nunique(), "-> months, not categories")
print("b11 missing and bord == 1:", n_first, "| b11 missing and bord > 1:", n_later, "| b11 present and bord == 1:", n_first_valid)
key_counts.append(["b11_missing_bord1", n_first])
key_counts.append(["b11_missing_bord_gt1", n_later])
key_counts.append(["b11_present_bord1", n_first_valid])

print()
for var in ["v113", "v116"]:
    print(var, "code counts:")
    print(cohort[var].value_counts().sort_index().to_string())
    key_counts.append([var + "_code97", int((cohort[var] == 97).sum())])
both97 = int(((cohort["v113"] == 97) & (cohort["v116"] == 97)).sum())
print("children with code 97 in BOTH v113 and v116:", both97, "| code 97 in v113:", int((cohort["v113"] == 97).sum()), "| in v116:", int((cohort["v116"] == 97).sum()))
print("-> same children in both variables: looks like a questionnaire-universe code, not a water/toilet type.")
print("   v135 (residence status) is not in the extract, so this cannot be cross-tabulated here.")

print()
for var in ["v701", "v705"]:
    print(var, "code counts (NaN shown separately):")
    print(cohort[var].value_counts(dropna=False).sort_index().to_string())
    key_counts.append([var + "_missing", int(cohort[var].isna().sum())])
print("v701 and v705 missing on the same rows:", bool((cohort["v701"].isna() == cohort["v705"].isna()).all()))
print("v501 (marital status) is not in the extract, so structural 'no partner' vs item non-response cannot be separated here.")

print()
print("v445 values >= 9990 (DHS flags):", int((cohort["v445"] >= 9990).sum()), "| NaN:", int(cohort["v445"].isna().sum()))
bmi = cohort["v445"][cohort["v445"] < 9990] / 100.0
print("BMI after /100:")
print(bmi.describe().round(2).to_string())
key_counts.append(["v445_flagged", int((cohort["v445"] >= 9990).sum())])
key_counts.append(["v445_nan", int(cohort["v445"].isna().sum())])

print()
band = pd.cut(cohort["b19"], [-1, 11, 23, 35, 47, 59])
print("m14 missing by child age band (audit only; m14/m15 are not predictors):")
print(cohort["m14"].isna().groupby(band, observed=False).mean().round(3).to_string())

print()
check_w = (cohort["v005"] / 1000000.0 - cohort["sample_weight"]).abs().max() if "sample_weight" in cohort.columns else float("nan")
print("sample_weight equals v005/1,000,000 (max abs difference):", check_w, "-> women's weight, not verified for child anthropometry (U4)")

In [ ]:
h.save_table(flow_df, h.OUTPUT_DIR + "/T1_sample_flow.csv")
h.save_table(pd.DataFrame(key_counts, columns=["check", "count"]), h.OUTPUT_DIR + "/n01_key_counts.csv")
h.save_table(cohort, h.PRIVATE_DIR + "/cohort_private.csv")
print("REMINDER: files in private/ contain survey identifiers. Do not upload or share them.")

### Gate - sign-off (you decide)
- [ ] I understand the outcome `stunting` is **taken as given**; D1 (hw70 rule) and U1 (sample-flow reconciliation) stay **OPEN**. If you can get the original KR file, run N01 later and compare.
- [ ] v021 = PSU and v022 = stratum (checked above).
- [ ] b11 is months; the code bins it (see notes: D14).
- [ ] v113/v116 code 97, v701/v705 missing: read the printout. The final coding is decided in N02.
- [ ] v445 has no flagged codes in this extract.
- [ ] I will mention in the thesis that the cohort could not be rebuilt from the original KR file (if that stays true).

In [ ]:
SIGN_OFF = False      # True only after the list above
if SIGN_OFF:
    f = open(h.OUTPUT_DIR + "/n01_signoff.txt", "w")
    f.write("N01B (extract mode) signed off. rows = " + str(len(cohort)) + ". OPEN: D1 outcome rule not verified, U1 sample flow not reconciled.\n")
    f.close()
    print("Gate passed provisionally. N02 may run. D1 and U1 remain open.")
else:
    print("Gate NOT passed yet.")